# Lesson 1.2: The Anatomy of a Perfect Prompt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Prompt-Engineering-Mastery/blob/main/tutorial/02-anatomy-perfect-prompt.ipynb)

If you write prompts like you're texting a friend or typing into Google, you'll get inconsistent, unpredictable results about 60% of the time. The other 40%? Lucky accidents you can't reproduce.

The fix is straightforward. Every reliable prompt — whether you're using ChatGPT, Claude, Gemini, or an open-source model — can be decomposed into **four structural pillars**. Master these four, and you stop guessing and start engineering.

**📍 Lesson Roadmap:**

| Section | Audience |
|:---|:---|
| 1. The 4-Pillar Framework | 🟢 Everyone |
| 2. Prompt Template Blueprint | 🟢 Everyone |
| 3. Model API Syntax | 🔷 Technical — Python SDK code for OpenAI, Claude, Gemini |
| 4. Framework in Action | 🟢 Everyone |
| Concept Check | 🟢 Everyone |

---

## 🟢 1. The 4-Pillar Prompt Framework

### Pillar 1: Role / Persona
**"Who is the model?"**

Setting a role constrains the model's vocabulary, tone, and domain expertise. Without a role, the model defaults to a generic, Wikipedia-flavored assistant voice.

```text
❌ Weak: "Help me write an email."
✅ Strong: "You are a senior B2B sales copywriter with 15 years of experience in SaaS."
```

> **💡 Pro Tip:** Be specific about the persona's experience level and domain. "You are a doctor" produces very different output than "You are a board-certified pediatric cardiologist at a teaching hospital." The more specific the role, the more constrained and useful the output.

### Pillar 2: Context / Background
**"What does the model need to know?"**

This is where you feed the raw materials — documents, data tables, customer profiles, code snippets, or constraints. Without context, the model hallucinates to fill the gaps.

### Pillar 3: Instructions / Task
**"What must it do?"**

Use **imperative verbs** — Analyze, Extract, Reformat, Compare, Classify, Summarize. Avoid vague instructions like "help me with" or "tell me about."

### Pillar 4: Constraints & Output Format
**"How must it return results?"**

Specify the exact output structure: JSON schema, bullet list, markdown table, character limit, language, things to exclude.

---

## 🟢 2. Prompt Template Blueprint

In a production pipeline, your prompt template is stored as a string with **variables** (placeholders) that you inject at runtime:

```text
ROLE: You are an expert {persona}.

CONTEXT DATA:
<context>
{input_data}
</context>

TASK:
{task_description}

CONSTRAINTS:
- Output format: {output_format}
- Do NOT include conversational preamble, disclaimers, or sign-off text.
- If any required field is missing from the context, output "MISSING" as the value.
```

> **⚠️ Common Mistake:** Notice the `<context>` XML tags wrapping the input data. This isn't optional decoration — it's a critical delimiter that prevents the model from confusing your data with your instructions. We'll go deeper into this in Lesson 3.1, but start using delimiters from day one.

---

## 🔷 3. How Each Model Handles the 4 Pillars

Each major provider structures the Role/Context/Task separation differently in their API. Understanding this is essential for writing portable prompts.

### OpenAI (GPT-4o)
OpenAI uses a **messages array** with explicit `system`, `user`, and `assistant` roles:

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o",
    temperature=0.0,
    messages=[
        {
            "role": "system",        # ← Pillar 1 (Role) + Pillar 4 (Constraints)
            "content": """You are a senior technical project manager.
Extract action items, assignees, and deadlines from meeting transcripts.
Output as a numbered markdown list. No preamble."""
        },
        {
            "role": "user",          # ← Pillar 2 (Context) + Pillar 3 (Task)
            "content": """Here is today's standup transcript:
<transcript>
Sarah: I'll finish the DB schema migration by Friday.
John: I'm coordinating with marketing on the Q3 launch. Need assets by Wednesday.
Lisa: Blocked on the API review — waiting for security team sign-off.
</transcript>

Extract all action items with assignees and deadlines."""
        }
    ]
)

### Anthropic (Claude 3.5 Sonnet, Claude 3 Opus)
Claude uses a dedicated `system` parameter (not inside the messages array) and strongly prefers **XML-tagged structure**:

In [ ]:
import anthropic
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-3-5-sonnet-latest",
    max_tokens=1024,
    temperature=0.0,
    system="""You are a senior technical project manager.
Extract action items, assignees, and deadlines from meeting transcripts.
Output as a numbered markdown list. No preamble.
If a deadline is not explicitly stated, write "No deadline specified".""",
    messages=[
        {
            "role": "user",
            "content": """Here is today's standup transcript:
<transcript>
Sarah: I'll finish the DB schema migration by Friday.
John: I'm coordinating with marketing on the Q3 launch. Need assets by Wednesday.
Lisa: Blocked on the API review — waiting for security team sign-off.
</transcript>

Extract all action items with assignees and deadlines."""
        }
    ]
)

### Google (Gemini 2.5 Pro, Gemini 2.5 Flash)
Gemini uses `system_instruction` for the role/constraints and `contents` for the user input:

In [ ]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model="gemini-2.5-pro",
    config={
        "system_instruction": """You are a senior technical project manager.
Extract action items, assignees, and deadlines from meeting transcripts.
Output as a numbered markdown list. No preamble.
If a deadline is not explicitly stated, write "No deadline specified".""",
        "temperature": 0.0,
    },
    contents="""Here is today's standup transcript:
<transcript>
Sarah: I'll finish the DB schema migration by Friday.
John: I'm coordinating with marketing on the Q3 launch. Need assets by Wednesday.
Lisa: Blocked on the API review — waiting for security team sign-off.
</transcript>

Extract all action items with assignees and deadlines."""
)

> **💡 Key Insight:** Notice that all three APIs physically separate the "system-level" instructions from the "user-level" input. This separation isn't just organizational — it's a **security boundary**. System instructions get higher priority than user messages, which is how you prevent prompt injection attacks (covered in Lesson 3.1).

---

## 🟢 4. The Framework in Action — Real Role Examples

<Tabs>
  <Tab label="📣 Marketer">
    * **Role:** "You are a direct-response growth marketer specializing in B2C e-commerce email campaigns."
    * **Context:** Product benefits sheet + target customer demographic profiles.
    * **Task:** "Generate 5 email subject line variants for the Spring Sale campaign."
    * **Constraints:** Each subject line must be under 50 characters, include one emoji, and avoid spam trigger words like "FREE" or "ACT NOW".
  </Tab>
  <Tab label="🔬 Researcher">
    * **Role:** "You are a biomedical research analyst with expertise in systematic reviews."
    * **Context:** Abstract from a clinical trial PDF.
    * **Task:** "Extract the study methodology, sample size, primary outcome, and p-value."
    * **Constraints:** Output as a Markdown table with columns [Field, Value, Page Reference]. If a field is not found, write "NOT REPORTED". Do not infer or assume values.
  </Tab>
  <Tab label="📋 Project Manager">
    * **Role:** "You are a certified PMP technical project manager at a mid-size SaaS company."
    * **Context:** Raw Zoom meeting transcript (30 min) containing side conversations and off-topic banter.
    * **Task:** "Filter the transcript to extract only actionable items. Build a task list."
    * **Constraints:** Output as JSON: `[{"task": "...", "assignee": "...", "deadline": "...", "priority": "high|medium|low"}]`. Ignore social chitchat.
  </Tab>
  <Tab label="💼 Analyst">
    * **Role:** "You are a senior IT operations analyst specializing in incident triage."
    * **Context:** 48 hours of server access logs (CSV format).
    * **Task:** "Identify the top 5 IP addresses with the highest error rates (HTTP 5xx)."
    * **Constraints:** Output as a markdown table sorted by error count descending. Include columns: IP, Error Count, Most Common Error Code, First Seen, Last Seen.
  </Tab>
</Tabs>

---

## 🟢 Concept Check

**Scenario:** You're building a customer feedback classifier. Your prompt says: *"Read the review and tell me if it's positive or negative."* The model keeps adding lengthy explanations and sometimes outputs "mixed" even though you only want binary classification. What's the most impactful fix?

* [x] **A)** Add explicit Constraints (Pillar 4): "Output only 'POSITIVE' or 'NEGATIVE'. No explanation. No other text."
* [ ] **B)** Increase the temperature to encourage more decisive responses.
* [ ] **C)** Switch to a larger model — smaller models can't do classification.
* [ ] **D)** Add "Please" to the prompt to make the model more compliant.

<details>
<summary><b>🔑 Click to Reveal Answer & Explanation</b></summary>

**Correct Answer: A**

**Explanation:** The original prompt violated Pillar 4 — it had no output constraints. Without explicit formatting rules, the model defaults to its training behavior: conversational, hedging, verbose. Adding strict constraints ("Output only X or Y, nothing else") is the single highest-leverage fix for most prompt failures. This is true across GPT, Claude, and Gemini alike.
</details>

---

## 📚 References & Further Reading

- **OpenAI Prompt Engineering Guide**: [platform.openai.com/docs/guides/prompt-engineering](https://platform.openai.com/docs/guides/prompt-engineering)
- **Anthropic's Prompt Engineering Guide**: [docs.anthropic.com/en/docs/build-with-claude/prompt-engineering](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering)
- **Google's Prompt Engineering Guide**: [ai.google.dev/gemini-api/docs/prompting-strategies](https://ai.google.dev/gemini-api/docs/prompting-strategies)
- **Zamfirescu-Pereira et al. (2023)**: *"Why Johnny Can't Prompt"* — research on how non-experts struggle with unstructured prompting

---

## 🚀 What's Next?

You've got the framework. But here's the thing — sometimes instructions alone aren't enough. If you need the model to match a very specific tone, formatting style, or classification scheme, you're better off *showing* it what you want rather than *telling* it. That's the power of few-shot prompting.

* [Lesson 2.1: Few-Shot Prompting & In-Context Learning →](./03-few-shot-prompting.mdx)